# Дом, который отстаёт от погоды

**Задача.** Дом обменивается теплом с улицей по закону Ньютона–Рихмана: чем больше разница температур, тем быстрее она уменьшается.

$$\frac{dT}{dt} = \frac{T_{\text{ул}}(t) - T}{\tau}, \qquad T_{\text{ул}}(t) = T_0 + a\cos\omega t .$$

Здесь $\tau$ — время подстройки: за время $\tau$ разница между домом и улицей уменьшается в $e \approx 2{,}7$ раза. У лёгкой постройки это часы, у кирпичного дома — сутки и больше. На паре получилось: внутри температура колеблется с той же частотой, но амплитуда меньше в $\cos\varphi$ раз и пик запаздывает на угол $\varphi$, где $\operatorname{tg}\varphi = \omega\tau$.

**Как запустить.** Нажмите **Ctrl+F9** (или «Среда выполнения» → «Выполнить всё»). На предупреждение «блокнот создан не Google» ответьте «Всё равно выполнить». Через несколько секунд ниже появится картинка с кнопками и ползунками.

Всё управление — кнопками и ползунками.

In [ ]:
#@title Вычисления { display-mode: "form" }
# =============================================================
#  ДОМ И УЛИЦА — движок: точное решение уравнения с пары.
#     dT/dt = (T_ул(t) - T) / tau,   T_ул = T0 + a cos(w (t - t_пик))
#  Интерфейс — в следующих ячейках. Здесь ничего не рисуется.
# =============================================================
import numpy as np


def fmt(x, d=1):
    return ('%.*f' % (d, x)).replace('.', ',')


# Два сценария: одно и то же уравнение, разные единицы.
SCEN = {
    'дом': dict(
        period=24.0, unit='ч', T0=20.0, a=10.0, t_peak=15.0, window=96.0,
        T_init=16.0, init_range=(0.0, 35.0),
        tau_min=0.2, tau_max=200.0, tau=24.0,
        presets=[('навес', 1.0), ('лёгкий дом', 5.0),
                 ('кирпичный', 24.0), ('толстостенный', 48.0)],
        outside='улица', inside='дом',
        t_out='снаружи', t_in='в доме', ratio='Колебания в доме меньше уличных'),
    'климат': dict(
        period=365.25, unit='сут', T0=6.0, a=15.0, t_peak=171.0,
        window=730.5, T_init=-8.0, init_range=(-25.0, 25.0),
        tau_min=3.0, tau_max=600.0, tau=33.0,
        presets=[('континент', 33.0), ('у моря', 100.0),
                 ('остров в океане', 200.0)],
        outside='солнце', inside='воздух',
        t_out='«по солнцу»', t_in='воздух', ratio='Колебания воздуха меньше «солнечных»'),
}


def omega(sc):
    return 2 * np.pi / sc['period']


def outside(sc, t):
    """Температура улицы (для климата — равновесная по солнцу)."""
    return sc['T0'] + sc['a'] * np.cos(omega(sc) * (t - sc['t_peak']))


def phase(sc, tau):
    return np.arctan(omega(sc) * tau)


def steady(sc, tau, t):
    """Периодический режим: T0 + a cos(phi) cos(w(t - t_пик) - phi)."""
    phi = phase(sc, tau)
    return sc['T0'] + sc['a'] * np.cos(phi) * np.cos(
        omega(sc) * (t - sc['t_peak']) - phi)


def house(sc, tau, T_init, t):
    """Общее решение = периодический режим + затухающая экспонента."""
    return steady(sc, tau, t) + (T_init - steady(sc, tau, 0.0)) * np.exp(-t / tau)


def slope(sc, tau, t, T):
    return (outside(sc, t) - T) / tau


def maxima(sc, tau, T_init, t):
    """Моменты, где производная меняет знак с + на -."""
    T = house(sc, tau, T_init, t)
    d = slope(sc, tau, t, T)
    i = np.where((d[:-1] > 0) & (d[1:] <= 0))[0]
    tm = t[i] + (t[i + 1] - t[i]) * d[i] / (d[i] - d[i + 1])
    return tm, house(sc, tau, T_init, tm)


def summary(sc, tau):
    phi = phase(sc, tau)
    return dict(wt=omega(sc) * tau, phi=np.degrees(phi),
                lag=phi / omega(sc), amp=sc['a'] * np.cos(phi))


def clock(h):
    h = h % 24.0
    hh, mm = int(h), int(round((h - int(h)) * 60))
    if mm == 60:
        hh, mm = (hh + 1) % 24, 0
    return '%d:%02d' % (hh, mm)


_MONTHS = [('января', 31), ('февраля', 28), ('марта', 31), ('апреля', 30),
           ('мая', 31), ('июня', 30), ('июля', 31), ('августа', 31),
           ('сентября', 30), ('октября', 31), ('ноября', 30), ('декабря', 31)]


def date(day):
    """День от 1 января (с нуля) -> '21 июля'."""
    d = int(round(day)) % 365
    for name, n in _MONTHS:
        if d < n:
            return '%d %s' % (d + 1, name)
        d -= n
    return '31 декабря'


def field_segments(sc, tau, t_rng, y_rng, nx=34, ny=16, frac=0.018):
    """Чёрточки поля направлений одинаковой длины на экране."""
    xr, yr = t_rng[1] - t_rng[0], y_rng[1] - y_rng[0]
    tt = np.linspace(t_rng[0], t_rng[1], nx + 2)[1:-1]
    yy = np.linspace(y_rng[0], y_rng[1], ny + 2)[1:-1]
    segs = []
    for t0 in tt:
        for y0 in yy:
            k = slope(sc, tau, t0, y0)
            dx, dy = 1.0 / xr, k / yr
            n = np.hypot(dx, dy)
            hx, hy = frac * dx / n * xr, frac * dy / n * yr
            segs.append([(t0 - hx, y0 - hy), (t0 + hx, y0 + hy)])
    return segs

In [ ]:
#@title Виджет: дом и улица { display-mode: "form" }
# =============================================================
#  ДОМ И УЛИЦА — интерфейс. Требует ячейку «Вычисления».
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

C_OUT, C_IN, INK, INK2, GRID = '#eb6834', '#2a78d6', '#0b0b0b', '#52514e', '#e6e5e0'
_SL = dict(style={'description_width': '120px'},
           layout=widgets.Layout(width='360px'))


class HouseApp:

    def __init__(self):
        self._busy = False
        self.sc = SCEN['дом']
        self._build_controls()
        self._build_figure()
        self._wire()
        display(self.panel, self.out, self.info)
        self._apply_scenario()

    # ---------------- органы управления ----------------
    def _build_controls(self):
        self.w_scen = widgets.ToggleButtons(
            options=[('Дом: сутки', 'дом'), ('Климат: год', 'климат')],
            value='дом')
        self.w_tau = widgets.FloatLogSlider(value=24.0, base=10, min=-0.7,
                                            max=2.3, step=0.01,
                                            description='τ, ч', readout_format='.3g',
                                            **_SL)
        self.w_T0 = widgets.FloatSlider(value=16.0, min=0.0, max=35.0, step=0.5,
                                        description='T в начале, °C', **_SL)
        self.w_max = widgets.Checkbox(value=True, indent=False,
                                      description='максимумы')
        self.w_field = widgets.Checkbox(value=False, indent=False,
                                        description='поле направлений')
        self.presets = widgets.HBox([])
        self.out = widgets.Output()
        self.info = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Дом и улица</h3>"),
            self.w_scen, self.presets,
            widgets.HBox([self.w_tau, self.w_T0]),
            widgets.HBox([self.w_max, self.w_field]),
        ])

    def _wire(self):
        self.w_scen.observe(lambda ch: self._apply_scenario(), names='value')
        for w in (self.w_tau, self.w_T0, self.w_max, self.w_field):
            w.observe(self._on_change, names='value')

    def _apply_scenario(self):
        sc = self.sc = SCEN[self.w_scen.value]
        buttons = []
        for name, tau in sc['presets']:
            b = widgets.Button(description='%s ≈ %s %s' % (name, fmt(tau, 0), sc['unit']),
                               layout=widgets.Layout(width='auto'))
            b.on_click(lambda _, v=tau: self._set(w_tau=v))
            buttons.append(b)
        self.presets.children = buttons
        self._busy = True
        try:
            self.w_tau.description = 'τ, ' + sc['unit']
            self.w_tau.min, self.w_tau.max = -10, 10
            self.w_tau.min = np.log10(sc['tau_min'])
            self.w_tau.max = np.log10(sc['tau_max'])
            self.w_tau.value = sc['tau']
            lo, hi = sc['init_range']
            self.w_T0.min, self.w_T0.max = -100, 100
            self.w_T0.min, self.w_T0.max = lo, hi
            self.w_T0.value = sc['T_init']
        finally:
            self._busy = False
        self._refresh()

    # ---------------- фигура строится ОДИН раз ----------------
    def _build_figure(self):
        self.fig, self.ax = plt.subplots(figsize=(11.2, 4.5), dpi=88)
        self.fig.subplots_adjust(left=0.06, right=0.89, top=0.92, bottom=0.12)
        plt.close(self.fig)
        ax = self.ax
        self.a_field = LineCollection([], colors='#bdbcb6', linewidths=0.9, zorder=1)
        ax.add_collection(self.a_field)
        self.l_out, = ax.plot([], [], color=C_OUT, lw=2, ls='--', zorder=3)
        self.l_in, = ax.plot([], [], color=C_IN, lw=2, zorder=4)
        self.p_max, = ax.plot([], [], 'o', ms=8, color=C_IN, mec='white',
                              mew=2, zorder=6)
        self.v_max = LineCollection([], colors=INK2, linewidths=0.8,
                                    linestyles=':', zorder=2)
        ax.add_collection(self.v_max)
        self.t_out = ax.text(0, 0, '', color=INK2, va='center', fontsize=10)
        self.t_in = ax.text(0, 0, '', color=INK2, va='center', fontsize=10)
        ax.set_ylabel('°C', color=INK2)
        ax.grid(axis='y', color=GRID, lw=0.8)
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        ax.tick_params(colors=INK2)

    def _axes_for(self, sc):
        ax = self.ax
        W = sc['window']
        ax.set_xlim(0, W)
        if sc is SCEN['дом']:
            ticks = np.arange(0, W + 1, 6)
            ax.set_xticks(ticks)
            ax.set_xticklabels(['%d:00' % (h % 24) for h in ticks])
        else:
            starts = np.cumsum([0] + [n for _, n in _MONTHS] * 2)[:-1]
            ax.set_xticks(starts[::3])
            ax.set_xticklabels(['1 янв', 'апр', 'июл', 'окт',
                                '1 янв', 'апр', 'июл', 'окт'])

    def render(self):
        sc, tau, T_init = self.sc, self.w_tau.value, self.w_T0.value
        t = np.linspace(0, sc['window'], 3000)
        To, Ti = outside(sc, t), house(sc, tau, T_init, t)
        ylo = min(To.min(), Ti.min()) - 2
        yhi = max(To.max(), Ti.max()) + 2
        ax = self.ax
        self._axes_for(sc)
        ax.set_ylim(ylo, yhi)
        self.l_out.set_data(t, To)
        self.l_in.set_data(t, Ti)
        x_lab = sc['window'] * 1.01
        self.t_out.set_position((x_lab, To[-1]))
        self.t_out.set_text(sc['outside'])
        self.t_in.set_position((x_lab, Ti[-1]))
        self.t_in.set_text(sc['inside'])
        if abs(To[-1] - Ti[-1]) < 0.06 * (yhi - ylo):      # не налезать друг на друга
            shift = 0.035 * (yhi - ylo) * (1 if Ti[-1] >= To[-1] else -1)
            self.t_in.set_position((x_lab, Ti[-1] + shift))
            self.t_out.set_position((x_lab, To[-1] - shift))
        if self.w_max.value:
            tm, Tm = maxima(sc, tau, T_init, t)
            self.p_max.set_data(tm, Tm)
            self.v_max.set_segments([[(x, ylo), (x, yhi)] for x in tm])
        else:
            self.p_max.set_data([], [])
            self.v_max.set_segments([])
        self.a_field.set_segments(field_segments(sc, tau, (0, sc['window']), (ylo, yhi))
                                  if self.w_field.value else [])
        s = summary(sc, tau)
        ax.set_title('τ = %s %s:  %s ±%s °C  →  %s ±%s °C,  пик на %s %s позже'
                     % (fmt(tau, 1 if tau < 10 else 0), sc['unit'], sc['t_out'],
                        fmt(sc['a'], 0), sc['t_in'], fmt(s['amp']),
                        fmt(s['lag'], 1 if s['lag'] < 10 else 0), sc['unit']),
                     loc='left', color=INK, fontsize=11)
        return self._info_html(sc, tau, s)

    def _info_html(self, sc, tau, s):
        if sc is SCEN['дом']:
            when = ('Снаружи жарче всего в %s, в доме — в <b>%s</b>.'
                    % (clock(sc['t_peak']), clock(sc['t_peak'] + s['lag'])))
        else:
            when = ('Солнце выше всего %s, воздух теплее всего <b>%s</b>.'
                    % (date(sc['t_peak']), date(sc['t_peak'] + s['lag'])))
        return ("<div style='font-size:14px; line-height:1.5'>"
                "%s %s в <b>%s</b> раза (это cos φ = %s); "
                "сдвиг фазы φ = %s°, ωτ = %s."
                "<br><span style='color:#52514e'>Начальная температура забывается за "
                "несколько τ: через 3τ = %s %s от неё остаётся 5 %%.</span></div>"
                % (when, sc['ratio'],
                   fmt(1 / np.cos(np.radians(s['phi'])), 2 if s['phi'] < 60 else 1),
                   fmt(np.cos(np.radians(s['phi'])), 2), fmt(s['phi'], 0),
                   fmt(s['wt'], 2), fmt(3 * tau, 0), sc['unit']))

    # ---------------- события ----------------
    def _refresh(self):
        self.info.value = self.render()
        with self.out:
            clear_output(wait=True)
            display(self.fig)

    def _on_change(self, change=None):
        if not self._busy:
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()


house_app = HouseApp()

### Что попробовать: нажимать по порядку

1. **«кирпичный ≈ 24 ч».** Снаружи $\pm 10$ °C, внутри всего $\pm 1{,}6$ °C, и жарче всего в доме не в 15:00, а в **20:24** — на 5,4 часа позже. Посмотрите на синие точки: **каждый максимум дома лежит ровно на пунктире улицы.** Почему — первый вопрос внизу.

2. **«навес ≈ 1 ч».** Постройка почти повторяет улицу: $\pm 9{,}7$ °C, запаздывание меньше часа. **«лёгкий дом ≈ 5 ч»** (как в задаче Бойса 2.3.14): $\pm 6{,}1$ °C, пик на 3,5 ч позже.

3. **«толстостенный ≈ 48 ч».** Колебания ещё вдвое меньше ($\pm 0{,}8$ °C), а запаздывание почти не выросло: 5,7 ч. Двигайте $\tau$ дальше вправо — больше **6 часов** (четверти суток) запаздывание не станет никогда.

4. **Ползунок «T в начале».** Первые сутки-двое дом «вспоминает», с чего начинал, потом выходит на один и тот же режим. Это и есть «общее = однородное + частное»: экспонента $Ce^{-t/\tau}$ гаснет, периодическая часть остаётся.

5. **«Климат: год».** То же уравнение, только период — год, а «улица» — температура, которую солнце установило бы мгновенно. На **«континенте»** ($\tau \approx 33$ сут) воздух теплее всего около **21 июля**, хотя солнце выше всего 21 июня. **«У моря»** ($\tau \approx 100$ сут) тёплый пик сдвигается на 21 августа, а размах уменьшается почти вдвое: вода — огромная теплоёмкость, «толстые стены» климата.

6. **Галочка «поле направлений».** Чёрточки горизонтальны ровно на пунктире: кривая улицы — это изоклина $m = 0$ с практики 3. Выше неё решение падает, ниже — растёт, поэтому дом всё время гонится за улицей и не догоняет.

---

## Тот же дом в лабораторной: RC-цепочка

Источник переменной ЭДС $\mathcal{E}_0\cos\omega t$, резистор $R$ и конденсатор $C$ соединены последовательно. По правилу Кирхгофа $IR + q/C = \mathcal{E}$, а ток $I = dq/dt$. Для напряжения на конденсаторе $U = q/C$ получается

$$RC\,\frac{dU}{dt} + U = \mathcal{E}_0\cos\omega t$$

— уравнение дома с $\tau = RC$. Что бы ни колебалось — дом, климат или напряжение, — ответ зависит только от одного числа $\omega\tau$: справа одна и та же кривая для всех.

In [ ]:
#@title Виджет: RC-цепочка { display-mode: "form" }
# =============================================================
#  RC-ЦЕПОЧКА — тот же дом. Требует ячейку «Вычисления».
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

C_OUT, C_IN, INK, INK2, GRID = '#eb6834', '#2a78d6', '#0b0b0b', '#52514e', '#e6e5e0'
_SLR = dict(style={'description_width': '90px'},
            layout=widgets.Layout(width='330px'))
REF = [('климат, континент', 2 * np.pi * 33 / 365.25),
       ('климат, у моря', 2 * np.pi * 100 / 365.25),
       ('кирпичный дом', 2 * np.pi)]


class RCApp:

    def __init__(self):
        self._busy = False
        self.w_R = widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=2, step=0.01,
                                          description='R, кОм', readout_format='.3g', **_SLR)
        self.w_C = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.01,
                                          description='C, мкФ', readout_format='.3g', **_SLR)
        self.w_f = widgets.FloatLogSlider(value=50.0, base=10, min=0, max=5, step=0.01,
                                          description='f, Гц', readout_format='.4g', **_SLR)
        buttons = []
        for label, f in (('лаба: 1 кОм, 1 мкФ, 50 Гц', 50.0), ('160 Гц', 160.0),
                         ('5 кГц', 5000.0)):
            b = widgets.Button(description=label, layout=widgets.Layout(width='auto'))
            b.on_click(lambda _, v=f: self._set(w_R=1.0, w_C=1.0, w_f=v))
            buttons.append(b)
        self.out, self.info = widgets.Output(), widgets.HTML()
        for w in (self.w_R, self.w_C, self.w_f):
            w.observe(self._on_change, names='value')
        display(widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>RC-цепочка</h3>"),
            widgets.HBox(buttons),
            widgets.HBox([self.w_R, self.w_C, self.w_f])]), self.out, self.info)
        self._build_figure()
        self._refresh()

    def _build_figure(self):
        self.fig, (self.ax, self.ax2) = plt.subplots(
            1, 2, figsize=(11.2, 4.2), dpi=88, gridspec_kw={'width_ratios': [1.5, 1]})
        self.fig.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.15, wspace=0.25)
        plt.close(self.fig)
        ax, ax2 = self.ax, self.ax2
        self.l_E, = ax.plot([], [], color=C_OUT, lw=2, ls='--', label='ЭДС  ℰ/ℰ₀')
        self.l_U, = ax.plot([], [], color=C_IN, lw=2, label='конденсатор  U/ℰ₀')
        ax.set_ylim(-1.15, 1.15)
        ax.legend(loc='lower left', frameon=False, fontsize=9)
        ax.grid(axis='y', color=GRID, lw=0.8)
        wt = np.logspace(-2, 2.3, 400)
        ax2.semilogx(wt, 1 / np.sqrt(1 + wt ** 2), color=INK2, lw=2)
        for name, x in REF:
            ax2.plot(x, 1 / np.sqrt(1 + x * x), 'o', ms=7, color='#b0afa8', mec='white', mew=1.5)
            ax2.text(x * 1.15, 1 / np.sqrt(1 + x * x) + 0.03, name, fontsize=8, color=INK2)
        self.p_now, = ax2.plot([], [], 'o', ms=10, color=C_IN, mec='white', mew=2, zorder=5)
        ax2.set_ylim(0, 1.1)
        ax2.set_xlabel('ωτ')
        ax2.set_title('во сколько раз гасятся колебания (cos φ)', loc='left',
                      color=INK, fontsize=10)
        ax2.grid(color=GRID, lw=0.8)
        for a in (ax, ax2):
            for s in ('top', 'right'):
                a.spines[s].set_visible(False)
            a.tick_params(colors=INK2)

    def render(self):
        R, C, f = self.w_R.value * 1e3, self.w_C.value * 1e-6, self.w_f.value
        tau, w = R * C, 2 * np.pi * f
        wt = w * tau
        phi = np.arctan(wt)
        T = 1.0 / f
        t = np.linspace(0, 3 * T, 900)
        self.l_E.set_data(t * 1e3, np.cos(w * t))
        self.l_U.set_data(t * 1e3, np.cos(phi) * np.cos(w * t - phi))
        self.ax.set_xlim(0, 3 * T * 1e3)
        self.ax.set_xlabel('t, мс')
        self.ax.set_title('три периода установившегося режима', loc='left',
                          color=INK, fontsize=10)
        self.p_now.set_data([wt], [np.cos(phi)])
        return ("<div style='font-size:14px; line-height:1.5'>τ = RC = <b>%s мс</b>, "
                "ωτ = <b>%s</b>. Амплитуда на конденсаторе — <b>%s ℰ₀</b>, "
                "фаза отстаёт на <b>%s°</b> (на %s мс).</div>"
                % (fmt(tau * 1e3, 3), fmt(wt, 2), fmt(np.cos(phi), 3),
                   fmt(np.degrees(phi), 1), fmt(phi / w * 1e3, 3)))

    def _refresh(self):
        self.info.value = self.render()
        with self.out:
            clear_output(wait=True)
            display(self.fig)

    def _on_change(self, change=None):
        if not self._busy:
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()


rc_app = RCApp()

### Что попробовать

1. **Три кнопки по очереди** — это числа из задачи Б6 листа. На 50 Гц конденсатор почти повторяет ЭДС (0,95 и 17°), на 5 кГц от сигнала остаётся 3 % и сдвиг почти четверть периода. Сравните со своими ответами.
2. **Найдите частоту, на которой амплитуда падает в $\sqrt2$ раз** (0,707). Чему там равно $\omega\tau$ и сдвиг фазы?
3. **Подберите f так, чтобы синяя точка села на «кирпичный дом».** На какой частоте RC-цепочка с $R = 1$ кОм, $C = 1$ мкФ ведёт себя ровно как кирпичный дом за сутки?

---

## Анзац: почему $T = u \cdot V$ работает

На паре искали решение в виде произведения $u\cdot V$, где $V$ решает однородное уравнение. Для уравнения $y' + p\,y = q(x)$ берём $V = e^{-px}$, тогда $u' = q\,e^{px}$. Виджет рисует все три множителя: $V$ гаснет, $u$ растёт, а произведение остаётся спокойным. (На практике 9 тот же ход назовут вариацией постоянной: $C$ в $Ce^{-px}$ становится функцией $u(x)$.)

In [ ]:
#@title Виджет: анзац u·V { display-mode: "form" }
# =============================================================
#  АНЗАЦ — три графика: V, u и произведение. Самостоятельная ячейка.
# =============================================================
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

C_OUT, C_IN, INK, INK2, GRID = '#eb6834', '#2a78d6', '#0b0b0b', '#52514e', '#e6e5e0'
PRESETS = {'здание: q = cos x': ('cos(x)', 1.0, 0.0),
           'лист 3, W22: q = x − 2': ('x - 2', 1.0, 0.0),
           'p < 0: решение растёт': ('cos(x)', -0.3, 0.0)}


def _q(expr, x):
    names = {'x': x, 'np': np, 'sin': np.sin, 'cos': np.cos, 'exp': np.exp,
             'pi': np.pi, 'abs': np.abs, 'sqrt': np.sqrt}
    try:
        r = eval(expr.replace('^', '**'), {'__builtins__': {}}, names)
        r = np.broadcast_to(np.asarray(r, dtype=float), x.shape).copy()
        return r if np.all(np.isfinite(r)) else None
    except Exception:
        return None


class AnsatzApp:

    def __init__(self):
        sl = dict(style={'description_width': '60px'}, layout=widgets.Layout(width='300px'))
        self.w_q = widgets.Text(value='cos(x)', description='q(x) =', continuous_update=False,
                                layout=widgets.Layout(width='300px'))
        self.w_p = widgets.FloatSlider(value=1.0, min=-0.5, max=2.0, step=0.1,
                                       description='p', **sl)
        self.w_u0 = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.25,
                                        description='u(0)', **sl)
        buttons = []
        for name, vals in PRESETS.items():
            b = widgets.Button(description=name, layout=widgets.Layout(width='auto'))
            b.on_click(lambda _, v=vals: self._preset(*v))
            buttons.append(b)
        self.out = widgets.Output()
        for w in (self.w_q, self.w_p, self.w_u0):
            w.observe(lambda ch: self._refresh(), names='value')
        self._busy = False
        display(widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>y' + p·y = q(x):  y = u·V</h3>"),
            widgets.HBox(buttons), widgets.HBox([self.w_q, self.w_p, self.w_u0])]), self.out)
        self._refresh()

    def _preset(self, q, p, u0):
        self._busy = True
        try:
            self.w_q.value, self.w_p.value, self.w_u0.value = q, p, u0
        finally:
            self._busy = False
        self._refresh()

    def _refresh(self):
        if self._busy:
            return
        x = np.linspace(0, 12, 1200)
        p, u0 = self.w_p.value, self.w_u0.value
        q = _q(self.w_q.value, x)
        with self.out:
            clear_output(wait=True)
            if q is None:
                print('Не получилось вычислить q(x). Пример записи: cos(x), x - 2, exp(-x)*sin(3*x)')
                return
            V = np.exp(-p * x)
            u = u0 + cumulative_trapezoid(q * np.exp(p * x), x, initial=0)
            y = u * V
            fig, ax = plt.subplots(1, 3, figsize=(11.2, 3.6), dpi=88)
            fig.subplots_adjust(left=0.05, right=0.99, top=0.85, bottom=0.14, wspace=0.32)
            ax[0].plot(x, V, color=INK2, lw=2)
            ax[0].set_title('V = e^(−px): решение однородного', loc='left', fontsize=10, color=INK)
            ax[1].plot(x, u, color=INK2, lw=2)
            ax[1].set_title("u: из u'·V = q", loc='left', fontsize=10, color=INK)
            ax[2].plot(x, q, color=C_OUT, lw=1.5, ls='--', label='q(x)')
            ax[2].plot(x, y, color=C_IN, lw=2, label='y = u·V')
            ax[2].legend(loc='upper right', frameon=False, fontsize=9)
            ax[2].set_title('произведение — решение', loc='left', fontsize=10, color=INK)
            for a in ax:
                a.grid(color=GRID, lw=0.8)
                a.set_xlabel('x')
                for s in ('top', 'right'):
                    a.spines[s].set_visible(False)
                if np.max(np.abs(a.get_ylim())) > 1e4:
                    a.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
            fig.text(0.355, 0.5, '×', fontsize=22, ha='center', va='center', color=INK2)
            fig.text(0.68, 0.5, '=', fontsize=22, ha='center', va='center', color=INK2)
            plt.show()


ansatz_app = AnsatzApp()

### Что попробовать

1. **«здание»**: $V$ гаснет как $e^{-x}$, $u$ растёт как $e^{x}$ — по отдельности обе функции «плохие», а произведение — спокойные колебания с той же частотой, что у $q$.
2. **«лист 3, W22»**: справа решение выходит на прямую $y = x - 3$ — ту самую изоклину-решение из листа 3. Поменяйте $u(0)$: меняется только начало, конец тот же.
3. **«p < 0»**: теперь $V$ растёт, а $u$ колеблется около постоянной — решение раскачивается. Притяжение к режиму было заслугой знака $p$: у дома $p = 1/\tau > 0$.
4. Впишите своё $q(x)$, например `exp(-x)*sin(3*x)` или `1`.

---

## По желанию: уравнение Бернулли может взорваться

№151 с пары: $y' + 2y = y^2e^x$. Замена $z = 1/y$ сделала его линейным, $z = e^x + Ce^{2x}$. Линейное уравнение никогда не уходит в бесконечность за конечное время, а $y = 1/z$ — может: там, где $z$ проходит через ноль.

In [ ]:
#@title По желанию: Бернулли и линейное { display-mode: "form" }
# =============================================================
#  №151:  z = e^x + C e^(2x)  (линейное),  y = 1/z  (Бернулли).
# =============================================================
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

C_OUT, C_IN, INK, INK2, GRID = '#eb6834', '#2a78d6', '#0b0b0b', '#52514e', '#e6e5e0'


def _fmt(x, d=2):
    return ('%.*f' % (d, x)).replace('.', ',')


class BernoulliApp:

    def __init__(self):
        self.w_C = widgets.FloatSlider(value=-0.5, min=-1.5, max=1.5, step=0.05,
                                       description='C', style={'description_width': '30px'},
                                       layout=widgets.Layout(width='360px'))
        self.out, self.info = widgets.Output(), widgets.HTML()
        self.w_C.observe(lambda ch: self._refresh(), names='value')
        display(self.w_C, self.out, self.info)
        self._refresh()

    def _refresh(self):
        C = self.w_C.value
        x = np.linspace(-2, 3, 2001)
        z = np.exp(x) + C * np.exp(2 * x)
        y = 1 / z
        y[np.abs(z) < 0.04] = np.nan
        with self.out:
            clear_output(wait=True)
            fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.2, 3.8), dpi=88)
            fig.subplots_adjust(left=0.06, right=0.98, top=0.86, bottom=0.14, wspace=0.22)
            a1.plot(x, z, color=INK2, lw=2)
            a1.axhline(0, color=INK2, lw=0.8)
            a1.set_ylim(-6, 6)
            a1.set_title('z = eˣ + C·e²ˣ  (линейное: гладкое везде)', loc='left',
                         fontsize=10, color=INK)
            a2.plot(x, y, color=C_IN, lw=2, label='y = 1/z')
            a2.axhline(0, color=C_OUT, lw=1.5, ls='--', label='y ≡ 0 (потеряли при делении)')
            a2.set_ylim(-6, 6)
            a2.set_title('y = 1/z  (Бернулли)', loc='left', fontsize=10, color=INK)
            a2.legend(loc='lower left', frameon=False, fontsize=9)
            msg = ('При C ≥ 0 знаменатель z не обращается в ноль: решение y '
                   'существует на всей прямой.')
            if C < 0:
                xs = np.log(-1 / C)
                for a in (a1, a2):
                    a.axvline(xs, color=INK2, lw=0.9, ls=':')
                msg = ('При C = %s знаменатель z проходит через ноль в x = ln(−1/C) ≈ %s: '
                       'там y уходит в бесконечность, хотя z остаётся гладким.'
                       % (_fmt(C), _fmt(xs)))
            for a in (a1, a2):
                a.grid(color=GRID, lw=0.8)
                a.set_xlabel('x')
                for s in ('top', 'right'):
                    a.spines[s].set_visible(False)
            plt.show()
        self.info.value = "<div style='font-size:14px'>%s</div>" % msg


bern_app = BernoulliApp()

---

## Вопросы к виджету

1. **Почему каждый максимум температуры дома лежит на кривой улицы?**

<details><summary>Ответ</summary>

В максимуме $T' = 0$, а из уравнения $T' = (T_{\text{ул}} - T)/\tau$ это значит $T = T_{\text{ул}}$. Это изоклина $m = 0$ с практики 3. Пока дом холоднее улицы, он греется; максимум наступает, когда улица, уже остывая, опускается до температуры дома. Отсюда и запаздывание: пик в доме всегда на спаде уличной температуры.

</details>

2. **Почему запаздывание никогда не больше четверти периода — 6 часов для суток, как ни утолщай стены?**

<details><summary>Ответ</summary>

$\varphi = \operatorname{arctg}(\omega\tau) < 90^\circ$ при любом $\tau$. По рисунку: максимум дома лежит на кривой улицы там, где она ещё выше средней, то есть не позже, чем через четверть периода после уличного пика. Толстые стены почти гасят колебания, а запаздывание упирается в 6 часов.

</details>

3. **На «кирпичном» колебания внутри $\pm 1{,}6$ °C, сдвиг $81^\circ$. Проверьте: $10\cos 81^\circ \approx 1{,}6$. Почему амплитуда равна $a\cos\varphi$?**

<details><summary>Ответ</summary>

Максимум дома лежит на кривой улицы (вопрос 1) и наступает через угол $\varphi$ после уличного пика. В этот момент улица показывает $T_0 + a\cos\varphi$ — это и есть максимум дома. Формула с пары, полученная интегрированием, говорит то же самое: $\dfrac{a}{\sqrt{1+\omega^2\tau^2}} = a\cos\varphi$ при $\operatorname{tg}\varphi = \omega\tau$.

</details>

4. **Какое $\tau$ нужно, чтобы пик в доме был ровно на 3 часа позже уличного?**

<details><summary>Ответ</summary>

3 часа — восьмая часть суток, $\varphi = 45^\circ$, $\omega\tau = 1$, $\tau = 24/(2\pi) \approx 3{,}8$ ч. Колебания внутри при этом меньше в $\sqrt2$ раз: $\pm 7{,}1$ °C.

</details>

5. **Климат: тёплый пик у моря в августе–сентябре, а не в июле. Что ещё должно отличаться у морского климата, если модель верна?**

<details><summary>Ответ</summary>

Размах. Чем больше $\tau$, тем больше запаздывание и тем меньше $\cos\varphi$. При $\tau = 100$ сут размах в $0{,}87/0{,}50 \approx 1{,}7$ раза меньше, чем на континенте. Так и есть: у моря зима мягче, а лето прохладнее — один параметр объясняет оба наблюдения.

</details>

6. **RC-цепочка: на какой частоте амплитуда падает в $\sqrt2$ раз?**

<details><summary>Ответ</summary>

При $\omega\tau = 1$: $f = 1/(2\pi RC) \approx 159$ Гц для $R = 1$ кОм, $C = 1$ мкФ. Фаза отстаёт на $45^\circ$. Эту частоту называют частотой среза фильтра.

</details>

7. **Бернулли: при каком $C$ решение $y = 1/(e^x + Ce^{2x})$ уходит в бесконечность при $x = 1$?**

<details><summary>Ответ</summary>

$e + Ce^2 = 0$, $C = -1/e \approx -0{,}37$. Поставьте ползунок на $-0{,}35$ или $-0{,}4$ и посмотрите, по какую сторону от $x = 1$ окажется пунктир.

</details>